# BanglaT5 Adaptive Explanations


## 1. Dependencies

Installs only what is missing. **PyTorch is never touched**, so your working CUDA build
stays as it is. If anything installs, restart the kernel before continuing.

In [1]:
import importlib, subprocess, sys

def have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

need = []
for mod, pkg in [("transformers", "transformers"), ("datasets", "datasets"),
                 ("sentencepiece", "sentencepiece"), ("captum", "captum"),
                 ("sklearn", "scikit-learn"), ("pandas", "pandas"),
                 ("numpy", "numpy"), ("matplotlib", "matplotlib"),
                 ("accelerate", "accelerate")]:
    if not have(mod):
        need.append(pkg)

if not have("torch"):
    raise SystemExit("PyTorch is missing. Install the CUDA build yourself, then re-run:\n"
                     "  pip install torch --index-url https://download.pytorch.org/whl/cu121")

if need:
    print("Installing:", ", ".join(need))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=False)
    print("\nDone. RESTART THE KERNEL, then run this notebook from the top.")
else:
    print("All dependencies present.")

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

C:\Users\mahmu\miniconda3\envs\smishing\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


All dependencies present.
torch 2.11.0+cu128 | CUDA True | NVIDIA GeForce RTX 5070


## 2. Configuration

In [2]:
import os, re, json, time, gc, random, unicodedata, hashlib, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch

# ---------------------------- paths ----------------------------------------
EXPL_CSV = Path("smishing_experiment/explanations/bangla_smishing_explanation_dataset.csv")
DEPLOY   = Path("deployment")
OLD_GEN  = Path("smishing_experiment/results/banglat5_generations.csv")   # optional baseline
OUT      = Path("explanation_fix"); OUT.mkdir(exist_ok=True)
for d in ["figures", "results", "models", "logs"]:
    (OUT / d).mkdir(exist_ok=True)

BANGLAT5_ID = "csebuetnlp/banglat5"
SEED = 42
LEVELS = ["static", "low", "medium", "high"]
LABELS = ["NORMAL", "PROMO", "SPAM"]
ID2LABEL = {0: "NORMAL", 1: "PROMO", 2: "SPAM"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

QUICK_TEST   = False    # True -> tiny budgets, proves the pipeline in ~15 min
RUN_TRAIN    = True
RUN_EVAL     = True
RUN_SAVE     = True
N_PRINT      = 50       # test SMS printed at the end (17 NORMAL / 17 PROMO / 16 SPAM)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16   = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

# ------------------- per-level training / generation configs ---------------
# HIGH gets a lower LR and more epochs: it is the longest target and was the least
# stable. LOW gets a hard short cap so it cannot drift into MEDIUM territory.
LEVEL_CFG = {
    "low":    dict(max_target=72,  epochs=8,  lr=3e-4, batch=8, accum=2,
                   gen=dict(min_new_tokens=10, max_new_tokens=70,  num_beams=4,
                            no_repeat_ngram_size=3, repetition_penalty=1.15,
                            length_penalty=1.0)),
    "medium": dict(max_target=144, epochs=8,  lr=3e-4, batch=8, accum=2,
                   gen=dict(min_new_tokens=28, max_new_tokens=140, num_beams=4,
                            no_repeat_ngram_size=3, repetition_penalty=1.20,
                            length_penalty=1.0)),
    "high":   dict(max_target=256, epochs=12, lr=2e-4, batch=4, accum=4,
                   gen=dict(min_new_tokens=55, max_new_tokens=250, num_beams=5,
                            no_repeat_ngram_size=4, repetition_penalty=1.25,
                            length_penalty=1.10)),
    "static": dict(max_target=144, epochs=8,  lr=3e-4, batch=8, accum=2,
                   gen=dict(min_new_tokens=25, max_new_tokens=140, num_beams=4,
                            no_repeat_ngram_size=3, repetition_penalty=1.20,
                            length_penalty=1.0)),
}
MAX_SOURCE = 384

if QUICK_TEST:
    for c in LEVEL_CFG.values():
        c["epochs"] = 1

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

set_seed()
print("device:", DEVICE, "| bf16:", BF16, "| seed:", SEED)
print("dataset :", EXPL_CSV)
print("deploy  :", DEPLOY.resolve())

device: cuda | bf16: True | seed: 42
dataset : smishing_experiment\explanations\bangla_smishing_explanation_dataset.csv
deploy  : C:\Users\mahmu\smishing-project\deployment


## 3. Load and inspect the real dataset

Nothing is assumed about the file — counts, splits, label balance, explanation lengths and
evidence quality are all measured before anything is changed.

In [3]:
if not EXPL_CSV.exists():
    cands = list(Path(".").rglob("bangla_smishing_explanation_dataset.csv"))
    if not cands:
        raise FileNotFoundError(
            f"Could not find {EXPL_CSV}. Put this notebook next to the "
            f"'smishing_experiment' folder, or edit EXPL_CSV in the config cell.")
    EXPL_CSV = cands[0]
    print("Found dataset at:", EXPL_CSV)

df = pd.read_csv(EXPL_CSV, encoding="utf-8-sig")
df.columns = [c.lstrip("\ufeff") for c in df.columns]
print("shape:", df.shape)
print("columns:", list(df.columns))

REQ = ["SMS", "label", "xai_evidence", "static_explanation",
       "adaptive_explanation_low", "adaptive_explanation_medium",
       "adaptive_explanation_high", "predicted_label", "confidence", "source_split"]
missing = [c for c in REQ if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for c in ["SMS", "label", "predicted_label", "source_split"]:
    df[c] = df[c].astype(str).str.strip()
df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce").fillna(0.0)
if "pred_matches_gold" in df.columns:
    df["pred_matches_gold"] = df["pred_matches_gold"].astype(str).str.upper().isin(["TRUE", "1", "YES"])
else:
    df["pred_matches_gold"] = df["label"] == df["predicted_label"]

LEVEL_COL = {"static": "static_explanation", "low": "adaptive_explanation_low",
             "medium": "adaptive_explanation_medium", "high": "adaptive_explanation_high"}
for c in LEVEL_COL.values():
    df[c] = df[c].fillna("").astype(str)

print("\nsource_split:\n", df["source_split"].value_counts().to_string())
print("\ngold label:\n", df["label"].value_counts().to_string())
print("\npredicted label:\n", df["predicted_label"].value_counts().to_string())
print(f"\npred_matches_gold: {df['pred_matches_gold'].mean()*100:.2f}% "
      f"({int((~df['pred_matches_gold']).sum())} disagreements)")
print(f"confidence: mean {df['confidence'].mean():.4f}  min {df['confidence'].min():.4f}")
df.head(3)

shape: (1641, 14)
columns: ['SMS', 'label', 'xai_evidence', 'static_explanation', 'adaptive_explanation_low', 'adaptive_explanation_medium', 'adaptive_explanation_high', 'label_id', 'predicted_label', 'confidence', 'pred_matches_gold', 'n_evidence', 'source_split', 'source_index']

source_split:
 source_split
train         1319
validation     170
test           152

gold label:
 label
NORMAL    673
SPAM      516
PROMO     452

predicted label:
 predicted_label
NORMAL    664
SPAM      523
PROMO     454

pred_matches_gold: 95.86% (68 disagreements)
confidence: mean 0.9398  min 0.4009


,SMS,label,xai_evidence,static_explanation,adaptive_explanation_low,adaptive_explanation_medium,adaptive_explanation_high,label_id,predicted_label,confidence,pred_matches_gold,n_evidence,source_split,source_index
0,🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁 ⭕️!!️ 🔠🔠🔠 🟣⚪️🟣⚪️🟣🟣⭐⭐ ⏰ কোড: NG88...,SPAM,"[{""text"": ""এক্সক্লুসিভ"", ""type"": ""linguistic_i...",বার্তাটিকে ঝুঁকিপূর্ণ বা প্রতারণামূলক হিসেবে চ...,বার্তাটিকে ঝুঁকিপূর্ণ বা প্রতারণামূলক হিসেবে চ...,বার্তাটিকে ঝুঁকিপূর্ণ বা প্রতারণামূলক হিসেবে চ...,বার্তাটিকে ঝুঁকিপূর্ণ বা প্রতারণামূলক হিসেবে চ...,2,SPAM,0.9783,True,6,train,5129
1,ও আচ্ছা তাহলে দুই মাস এবং তিন মাসের কোর্স ফি ট...,NORMAL,"[{""text"": ""মাস"", ""type"": ""linguistic_indicator...",এটি একটি সাধারণ বার্তা। মডেলের আস্থা 99.0%। সি...,এটি একটি সাধারণ বার্তা। বার্তাটিতে “মাস” এবং “...,এটি একটি সাধারণ বার্তা। বার্তায় যে অংশগুলো সি...,এটি একটি সাধারণ বার্তা। মডেলের ক্যালিব্রেটেড আ...,0,NORMAL,0.9896,True,6,train,10279
2,আজই শেষ দিন ! ৮জিবি+২০০মিনিট -২০০টাকা (৩০দিন)ব...,PROMO,"[{""text"": ""৮জিবি+২০০মিনিট"", ""type"": ""promo_off...",এটি একটি প্রচারমূলক (প্রমোশনাল) বার্তা। মডেলের...,এটি একটি প্রচারমূলক (প্রমোশনাল) বার্তা। বার্তা...,এটি একটি প্রচারমূলক (প্রমোশনাল) বার্তা। বার্তা...,এটি একটি প্রচারমূলক (প্রমোশনাল) বার্তা। মডেলের...,1,PROMO,0.8781,True,6,train,1556


In [4]:
_SENT_SPLIT = re.compile(r"(?:।+|[!?]+|\.(?=\s|$))")
def sent_count(t):
    return len([x for x in _SENT_SPLIT.split(str(t)) if x.strip()])
def word_count(t):
    return len([w for w in re.split(r"\s+", str(t).strip()) if w])

print(f"{'level':8s} {'chars(med)':>11s} {'words(med)':>11s} {'sents(med)':>11s} "
      f"{'sents(max)':>11s} {'empty':>7s}")
print("-" * 66)
for lv, col in LEVEL_COL.items():
    s = df[col]
    print(f"{lv:8s} {s.str.len().median():11.0f} "
          f"{s.map(word_count).median():11.0f} {s.map(sent_count).median():11.0f} "
          f"{s.map(sent_count).max():11.0f} {int((s.str.len()==0).sum()):7d}")

# how many DISTINCT opening sentences does each level use?
print("\nDistinct opening sentences per level (why the CSV looks repetitive in Excel):")
for lv, col in LEVEL_COL.items():
    op = df[col].map(lambda t: _SENT_SPLIT.split(str(t))[0].strip()[:40])
    print(f"  {lv:8s}: {op.nunique():3d} distinct openers")
same = (df[LEVEL_COL['low']].map(lambda t: _SENT_SPLIT.split(str(t))[0][:25]) ==
        df[LEVEL_COL['high']].map(lambda t: _SENT_SPLIT.split(str(t))[0][:25])).mean()
print(f"\n  LOW and HIGH share the same opening sentence on {same*100:.1f}% of rows.")
print("  -> that is the Excel 'repetition', not degenerate text. Fixed by the new realiser.")

level     chars(med)  words(med)  sents(med)  sents(max)   empty
------------------------------------------------------------------
static           304          42           5           6       0
low              193          27           3           3       0
medium           340          47           4           5       0
high             625          85           6           7       0

Distinct opening sentences per level (why the CSV looks repetitive in Excel):
  static  :  11 distinct openers
  low     :   9 distinct openers
  medium  :   9 distinct openers
  high    :  10 distinct openers

  LOW and HIGH share the same opening sentence on 99.9% of rows.
  -> that is the Excel 'repetition', not degenerate text. Fixed by the new realiser.


## 4. Diagnosis — measuring both problems on the real data

`detect_asserted_class` reads what an explanation *claims* the message is, handling Bangla
negation (`… প্রতারণামূলক হয় না` asserts nothing). Comparing that to `predicted_label`
gives the contradiction rate; comparing it to `label` shows whether contradictions track
the classifier's errors, which is the signature of the train/inference mismatch.

In [5]:
CLASS_MARKERS = {
    "SPAM":   ["প্রতারণামূলক", "স্মিশিং", "প্রতারণার চেষ্টা", "ঝুঁকিপূর্ণ", "ভুয়া",
               "জালিয়াতি", "প্রতারক", "ফিশিং", "প্রতারণা"],
    "PROMO":  ["প্রচারমূলক", "প্রমোশনাল", "অফার সংক্রান্ত", "বিজ্ঞাপন", "প্যাকেজ সংক্রান্ত"],
    "NORMAL": ["সাধারণ বার্তা", "স্বাভাবিক ও নিরাপদ", "স্বাভাবিক বার্তা", "নিরাপদ বার্তা",
               "তথ্যমূলক বার্তা", "প্রতারণার লক্ষণ পাওয়া যায়নি"],
}
NEGATORS = ["নয়", "হয় না", "পাওয়া যায়নি", "যায়নি", "গণ্য করা হয়নি", "নেই", "না।", "হয়নি"]

def detect_asserted_class(text, window=34):
    """What class does this text CLAIM? None if it makes no class claim.
    A negated mention ('... is not fraudulent') is not a claim."""
    text = str(text)
    hits = Counter()
    for cls, marks in CLASS_MARKERS.items():
        for m in marks:
            for hit in re.finditer(re.escape(m), text):
                tail = text[hit.end():hit.end() + window]
                if any(neg in tail for neg in NEGATORS):
                    continue
                hits[cls] += 1
    if not hits:
        return None
    top = hits.most_common()
    if len(top) > 1 and top[0][1] == top[1][1]:
        return None
    return top[0][0]

# sanity checks
_cases = [("এই বার্তাটি প্রতারণামূলক (স্মিশিং) বলে শনাক্ত করা হয়েছে।", "SPAM"),
          ("এটি একটি প্রচারমূলক (প্রমোশনাল) বার্তা।", "PROMO"),
          ("এই এসএমএসটিতে প্রতারণার লক্ষণ পাওয়া যায়নি।", "NORMAL"),
          ("শুধু ডায়াল কোড থাকলেই বার্তাটি প্রতারণামূলক হয় না।", None),
          ("ডায়াল কোডটি অফারের অংশ।", None)]
print("detect_asserted_class sanity check")
for t, exp in _cases:
    got = detect_asserted_class(t)
    print(f"  {'OK ' if got == exp else 'FAIL'} expected={str(exp):7s} got={str(got):7s} | {t[:52]}")
assert all(detect_asserted_class(t) == e for t, e in _cases), "assertion detector is wrong"

detect_asserted_class sanity check
  OK  expected=SPAM    got=SPAM    | এই বার্তাটি প্রতারণামূলক (স্মিশিং) বলে শনাক্ত করা হয
  OK  expected=PROMO   got=PROMO   | এটি একটি প্রচারমূলক (প্রমোশনাল) বার্তা।
  OK  expected=NORMAL  got=NORMAL  | এই এসএমএসটিতে প্রতারণার লক্ষণ পাওয়া যায়নি।
  OK  expected=None    got=None    | শুধু ডায়াল কোড থাকলেই বার্তাটি প্রতারণামূলক হয় না।
  OK  expected=None    got=None    | ডায়াল কোডটি অফারের অংশ।


In [6]:
GARBAGE_COLON = re.compile(r"\d+\s*:\s*\d+\s*:\s*\d+")          # 173:37298:1993
GARBAGE_RUN   = re.compile(r"(?:\b[\d.,%]+\b[\s,;:]*){4,}")      # 4+ numbers in a row
LATIN_RE      = re.compile(r"[A-Za-z]")
BENGALI_RE    = re.compile(r"[\u0980-\u09FF]")
PROMPT_LEAK   = ["SMS:", "LABEL:", "XAI EVIDENCE:", "EXPLANATION TYPE:", "ব্যাখ্যা:" + "\n"]

def digit_ratio(t):
    t = str(t)
    return sum(ch.isdigit() for ch in t) / max(1, len(t))

def latin_ratio(t):
    t = str(t)
    n_lat = len(LATIN_RE.findall(t)); n_bn = len(BENGALI_RE.findall(t))
    return n_lat / max(1, n_lat + n_bn)

def repeat_ratio(t, n=4):
    w = str(t).split()
    if len(w) < n * 2:
        return 0.0
    g = [" ".join(w[i:i+n]) for i in range(len(w)-n+1)]
    return 1 - len(set(g)) / len(g)

def garbage_codes(t):
    t = str(t); c = []
    if not t.strip():                      c.append("empty")
    if GARBAGE_COLON.search(t):            c.append("colon_number_garbage")
    if GARBAGE_RUN.search(t):              c.append("numeric_run")
    if digit_ratio(t) > 0.12:              c.append(f"digit_ratio:{digit_ratio(t):.2f}")
    if latin_ratio(t) > 0.35:              c.append(f"latin_ratio:{latin_ratio(t):.2f}")
    if repeat_ratio(t) > 0.30:             c.append(f"repetition:{repeat_ratio(t):.2f}")
    if "\ufffd" in t:                      c.append("broken_unicode")
    if any(p in t for p in PROMPT_LEAK):   c.append("prompt_leak")
    return c

rows = []
for _, r in df.iterrows():
    for lv, col in LEVEL_COL.items():
        txt = r[col]
        asserted = detect_asserted_class(txt)
        rows.append({
            "level": lv, "gold": r["label"], "pred": r["predicted_label"],
            "pred_matches_gold": r["pred_matches_gold"],
            "asserted": asserted,
            "contradicts_classifier": (asserted is not None and asserted != r["predicted_label"]),
            "no_class_claim": asserted is None,
            "garbage": ";".join(garbage_codes(txt)),
            "is_garbage": len(garbage_codes(txt)) > 0,
            "digit_ratio": digit_ratio(txt),
            "sents": sent_count(txt),
        })
diag = pd.DataFrame(rows)

print("=" * 74); print("PROBLEM 1 — classifier/explanation contradiction (OLD data)"); print("=" * 74)
print(diag.groupby("level")["contradicts_classifier"].mean().mul(100).round(2).to_string()
      + "   (% of rows)")
overall = diag["contradicts_classifier"].mean()
print(f"\noverall contradiction rate: {overall*100:.2f}%")

print("\nContradiction rate split by whether the classifier agreed with the gold label:")
pt = diag.groupby("pred_matches_gold")["contradicts_classifier"].agg(["mean", "size"])
for k, v in pt.iterrows():
    print(f"  pred_matches_gold={str(k):5s}  n={int(v['size']):6d}  "
          f"contradiction={v['mean']*100:6.2f}%")
print("\nIf contradictions are concentrated where pred != gold, the cause is the")
print("train/inference label mismatch described at the top, not model weakness.")

print("\n" + "=" * 74); print("PROBLEM 2 — malformed / garbage targets (OLD data)"); print("=" * 74)
g = diag.groupby("level").agg(garbage_rate=("is_garbage", "mean"),
                              mean_digit_ratio=("digit_ratio", "mean"),
                              median_sents=("sents", "median"))
print((g.assign(garbage_rate=lambda d: (d.garbage_rate*100).round(2))).to_string())
codes = Counter(c for s in diag["garbage"] if s for c in s.split(";"))
print("\nmost common defect codes:")
for c, n in codes.most_common(10):
    print(f"  {c:28s} {n}")

diag.to_csv(OUT / "results" / "old_target_diagnosis.csv", index=False, encoding="utf-8-sig")
OLD_DIAG = {"contradiction_rate_overall": float(overall),
            "contradiction_by_level": diag.groupby("level")["contradicts_classifier"].mean().to_dict(),
            "garbage_by_level": diag.groupby("level")["is_garbage"].mean().to_dict(),
            "mean_digit_ratio_by_level": diag.groupby("level")["digit_ratio"].mean().to_dict()}

PROBLEM 1 — classifier/explanation contradiction (OLD data)
level
high      0.0
low       0.0
medium    0.0
static    0.0   (% of rows)

overall contradiction rate: 0.00%

Contradiction rate split by whether the classifier agreed with the gold label:
  pred_matches_gold=False  n=   272  contradiction=  0.00%
  pred_matches_gold=True   n=  6292  contradiction=  0.00%

If contradictions are concentrated where pred != gold, the cause is the
train/inference label mismatch described at the top, not model weakness.

PROBLEM 2 — malformed / garbage targets (OLD data)
        garbage_rate  mean_digit_ratio  median_sents
level                                               
high            0.61          0.030586           6.0
low             0.43          0.011743           3.0
medium          0.61          0.007351           4.0
static          0.67          0.019404           5.0

most common defect codes:
  colon_number_garbage         34
  numeric_run                  3
  latin_ratio:0.81   

## 5. Evidence parsing, cleaning and re-ranking

`xai_evidence` is parsed, every span is re-verified against the SMS, and weak generic
tokens (`ও`, `এবং`, `মাস`, `are`, `how`, `the`, …) are dropped. Meaningful types — URL,
phone, money, USSD, sensitive-info request, urgency, impersonation, reward, promo offer —
are promoted above bare `linguistic_indicator` spans regardless of raw attribution, so the
sentences the model learns to write are anchored on real signals.

In [7]:
_ZW = dict.fromkeys(map(ord, "\u200b\u200c\u200d\ufeff"), None)
def normalize_bn(t):
    t = "" if t is None else str(t)
    t = unicodedata.normalize("NFKC", t).translate(_ZW)
    return re.sub(r"\s+", " ", t).strip()

URL_RE   = re.compile(r"(https?://\S+|www\.\S+|\b[a-z0-9\-]+\.(?:com|net|org|xyz|top|info|link|ly|bd|gov\.bd|co)\b\S*)", re.I)
USSD_RE  = re.compile(r"\*[\d\*#]{2,}#")
PHONE_RE = re.compile(r"(\+?880\d{8,10}|\b01[3-9]\d{8}\b|\b\d{11}\b)")
MONEY_RE = re.compile(r"(৳\s?[\d০-৯,.]+|[\d০-৯,.]+\s?(?:টাকা|tk|taka|৳)|moneytoken)", re.I)

EVIDENCE_TYPE_BN = {
    "suspicious_url": "সন্দেহজনক লিংক", "url": "ওয়েব লিংক",
    "urgency": "তাড়াহুড়ার ভাষা", "sensitive_info_request": "গোপন তথ্য চাওয়া",
    "financial_request": "আর্থিক লেনদেনের অনুরোধ", "impersonation": "পরিচয় ভাঁড়ানোর ইঙ্গিত",
    "reward_claim": "পুরস্কার বা উপহারের দাবি", "threat_warning": "ভয় দেখানো বা সতর্কবার্তা",
    "suspicious_action": "নির্দিষ্ট পদক্ষেপ নিতে বলা", "phone_number": "ফোন নম্বর",
    "ussd_code": "ডায়াল কোড", "money_amount": "টাকার অঙ্ক",
    "promo_offer": "অফার বা প্যাকেজের তথ্য", "linguistic_indicator": "ভাষাগত ইঙ্গিত",
}

# meaningful types outrank generic linguistic hits, whatever the raw attribution said
TYPE_PRIORITY = {
    "suspicious_url": 10, "sensitive_info_request": 10, "financial_request": 9,
    "reward_claim": 9, "impersonation": 8, "threat_warning": 8, "urgency": 7,
    "suspicious_action": 7, "url": 6, "phone_number": 6, "ussd_code": 5,
    "money_amount": 5, "promo_offer": 4, "linguistic_indicator": 1,
}

WEAK_TOKENS = {
    "ও", "এবং", "কিন্তু", "তবে", "এই", "সেই", "একটি", "এটি", "আপনি", "আপনার", "করুন",
    "জন্য", "থেকে", "মাস", "টি", "যে", "যা", "হবে", "হয়", "করা", "সঙ্গে", "মধ্যে",
    "are", "how", "the", "is", "to", "for", "and", "you", "your", "a", "of", "in",
    "on", "it", "be", "as", "at", "or", "an",
}


_SPAN_EDGE = " \t\n\r\u201c\u201d\"'()[]{}<>«»।!?,;:…\u0964"
_INTERNAL_TERM = re.compile(r"[।!?]")

def sanitize_span(txt, sms):
    """Trim edge punctuation, keep it a literal substring of the SMS, and reject spans
    that still carry a sentence terminator inside them."""
    t = str(txt).strip(_SPAN_EDGE).rstrip(".").strip()
    if not t or len(t) < 2:
        return None
    if _INTERNAL_TERM.search(t):          # e.g. "দিন!করুন" — cannot be quoted safely
        return None
    if t not in sms:                       # trimming must not break the substring guarantee
        return None
    return t
    
def clean_evidence(ev_json, sms):
    """Parse, verify against the SMS, drop weak spans, re-rank by type then importance."""
    try:
        ev = json.loads(ev_json) if isinstance(ev_json, str) else (ev_json or [])
    except Exception:
        return []
    out, seen = [], set()
    for e in ev:
        txt = sanitize_span(normalize_bn(e.get("text", "")), sms)
        if not txt or txt in seen:
            continue
        typ = e.get("type") or "linguistic_indicator"
        low = txt.lower()
        if typ == "linguistic_indicator" and (low in WEAK_TOKENS or len(txt) <= 2):
            continue
        if re.fullmatch(r"[\d০-৯,.%]+", txt):                # a bare number is not evidence
            continue
        seen.add(txt)
        out.append({"text": txt, "type": typ,
                    "type_bn": e.get("type_bn") or EVIDENCE_TYPE_BN.get(typ, typ),
                    "importance": float(e.get("importance", 0.0)),
                    "priority": TYPE_PRIORITY.get(typ, 1)})
    out.sort(key=lambda d: (-d["priority"], -d["importance"]))
    return out

df["SMS"] = df["SMS"].map(normalize_bn)
df["evidence"] = [clean_evidence(r["xai_evidence"], r["SMS"]) for _, r in df.iterrows()]
df["n_ev_clean"] = df["evidence"].map(len)

before = df["xai_evidence"].map(lambda s: len(json.loads(s)) if isinstance(s, str) and s.strip().startswith("[") else 0)
print(f"evidence spans: before mean {before.mean():.2f} -> after cleaning {df['n_ev_clean'].mean():.2f}")
print(f"rows left with zero usable evidence: {int((df['n_ev_clean']==0).sum())} "
      f"({(df['n_ev_clean']==0).mean()*100:.1f}%)")
print("\nevidence type mix after cleaning:")
tc = Counter(e["type"] for lst in df["evidence"] for e in lst)
for t, n in tc.most_common():
    print(f"  {t:24s} {EVIDENCE_TYPE_BN.get(t,t):26s} {n}")

evidence spans: before mean 5.35 -> after cleaning 4.65
rows left with zero usable evidence: 1 (0.1%)

evidence type mix after cleaning:
  linguistic_indicator     ভাষাগত ইঙ্গিত              5856
  promo_offer              অফার বা প্যাকেজের তথ্য     430
  ussd_code                ডায়াল কোড                 340
  financial_request        আর্থিক লেনদেনের অনুরোধ     266
  urgency                  তাড়াহুড়ার ভাষা           138
  reward_claim             পুরস্কার বা উপহারের দাবি   121
  money_amount             টাকার অঙ্ক                 111
  sensitive_info_request   গোপন তথ্য চাওয়া           97
  suspicious_action        নির্দিষ্ট পদক্ষেপ নিতে বলা 67
  phone_number             ফোন নম্বর                  55
  impersonation            পরিচয় ভাঁড়ানোর ইঙ্গিত    47
  url                      ওয়েব লিংক                 46
  threat_warning           ভয় দেখানো বা সতর্কবার্তা  37
  suspicious_url           সন্দেহজনক লিংক             18


## 6. The rewritten realiser — targets regenerated from the PREDICTED label

Three rules, each aimed at a measured defect:

1. **Every target explains `predicted_label`.** The classifier is the source of truth; the
   gold label never enters a target or an input.
2. **No generated number ever appears.** Confidence becomes qualitative Bangla. Importance
   is expressed by ordering and wording. Digits survive only inside a quoted evidence span
   that occurs in the SMS. This is what removes `173:37298:1993` at the source.
3. **No ML jargon in HIGH** — no gradients, logits, embeddings, token-level percentages.
   HIGH is deeper *about the message*, not about the model's internals.

Each level has its own opener bank and its own sentence plan, so the four columns are
visibly different.

In [10]:
LABEL_BN = {"SPAM": "প্রতারণামূলক (স্মিশিং) বার্তা",
            "PROMO": "প্রচারমূলক বার্তা",
            "NORMAL": "সাধারণ বার্তা"}

OPENERS = {   # distinct per level so the four columns never look identical
 "low": {
   "SPAM":  ["সতর্ক থাকুন — এই এসএমএসটি প্রতারণামূলক।", "এই বার্তাটি প্রতারণামূলক মনে হচ্ছে।"],
   "PROMO": ["এটি একটি প্রচারমূলক বার্তা।", "এই এসএমএসটি একটি অফারের বার্তা।"],
   "NORMAL":["এটি একটি সাধারণ বার্তা।", "এই এসএমএসটি স্বাভাবিক মনে হচ্ছে।"]},
 "medium": {
   "SPAM":  ["বার্তাটিকে প্রতারণামূলক (স্মিশিং) হিসেবে শনাক্ত করা হয়েছে।",
             "যাচাই শেষে বার্তাটিকে প্রতারণামূলক বলে চিহ্নিত করা হয়েছে।"],
   "PROMO": ["বার্তাটিকে প্রচারমূলক বা অফারের বার্তা হিসেবে শনাক্ত করা হয়েছে।",
             "যাচাই শেষে এটিকে একটি প্রমোশনাল বার্তা হিসেবে চিহ্নিত করা হয়েছে।"],
   "NORMAL":["বার্তাটিকে সাধারণ বা তথ্যমূলক বার্তা হিসেবে শনাক্ত করা হয়েছে।",
             "যাচাই শেষে এটিকে একটি স্বাভাবিক বার্তা হিসেবে চিহ্নিত করা হয়েছে।"]},
 "high": {
   "SPAM":  ["বিস্তারিত বিশ্লেষণে বার্তাটি প্রতারণামূলক (স্মিশিং) হিসেবে চিহ্নিত হয়েছে।",
             "সামগ্রিক বিশ্লেষণ অনুযায়ী এই এসএমএসটি একটি প্রতারণার চেষ্টা।"],
   "PROMO": ["বিস্তারিত বিশ্লেষণে বার্তাটি একটি প্রচারমূলক বার্তা হিসেবে চিহ্নিত হয়েছে।",
             "সামগ্রিক বিশ্লেষণ অনুযায়ী এটি একটি বাণিজ্যিক অফারের বার্তা।"],
   "NORMAL":["বিস্তারিত বিশ্লেষণে বার্তাটি সাধারণ বার্তা হিসেবে চিহ্নিত হয়েছে।",
             "সামগ্রিক বিশ্লেষণ অনুযায়ী এই এসএমএসটি স্বাভাবিক ও তথ্যমূলক।"]},
 "static": {
   "SPAM":  ["শ্রেণিবিন্যাস অনুযায়ী এটি একটি প্রতারণামূলক (স্মিশিং) বার্তা।"],
   "PROMO": ["শ্রেণিবিন্যাস অনুযায়ী এটি একটি প্রচারমূলক বার্তা।"],
   "NORMAL":["শ্রেণিবিন্যাস অনুযায়ী এটি একটি সাধারণ বার্তা।"]},
}

CONF_BN = lambda c: ("মডেলটি এই সিদ্ধান্তে বেশ নিশ্চিত" if c >= 0.90 else
                     "মডেলটি মোটামুটি নিশ্চিত" if c >= 0.70 else
                     "মডেলটি সম্পূর্ণ নিশ্চিত নয়")

WHY = {  # SPAM wording vs neutral wording — the PROMO/SPAM separation lives here
 "suspicious_url": ("অচেনা বা সংক্ষিপ্ত লিংক ভুয়া ওয়েবসাইটে নিয়ে গিয়ে তথ্য চুরি করতে পারে",
                    "লিংকটি বার্তার অংশ হিসেবে দেওয়া হয়েছে"),
 "url": ("লিংকে ক্লিক করালে ভুয়া পেজে নেওয়ার ঝুঁকি থাকে", "প্রতিষ্ঠানের ওয়েব ঠিকানা দেওয়া হয়েছে"),
 "urgency": ("তাড়াহুড়া তৈরি করে ভাবার সময় না দেওয়া প্রতারকদের পরিচিত কৌশল",
             "অফারের সময়সীমা বোঝাতে এমন শব্দ স্বাভাবিকভাবেই ব্যবহৃত হয়"),
 "sensitive_info_request": ("পিন, ওটিপি বা পাসওয়ার্ড কোনো প্রতিষ্ঠান এসএমএসে চায় না",
                            "যাচাই সংক্রান্ত শব্দ ব্যবহার করা হয়েছে"),
 "financial_request": ("টাকা পাঠাতে বলা প্রতারণার একটি বড় লক্ষণ",
                       "রিচার্জ বা পেমেন্ট সংক্রান্ত স্বাভাবিক তথ্য"),
 "impersonation": ("পরিচিত প্রতিষ্ঠানের নাম ব্যবহার করে বিশ্বাস অর্জনের চেষ্টা করা হয়",
                   "প্রতিষ্ঠানের নাম উল্লেখ করা হয়েছে"),
 "reward_claim": ("না চাইতেই পুরস্কার জেতার দাবি প্রায় সবসময়ই ভুয়া",
                  "উপহার বা ছাড়ের প্রস্তাব দেওয়া হয়েছে"),
 "threat_warning": ("অ্যাকাউন্ট বন্ধের ভয় দেখিয়ে দ্রুত সাড়া দিতে বাধ্য করা হয়",
                    "মেয়াদ বা শর্ত সংক্রান্ত তথ্য দেওয়া হয়েছে"),
 "suspicious_action": ("ক্লিক বা ফরম পূরণের নির্দেশ দিয়ে তথ্য হাতিয়ে নেওয়া হয়",
                       "গ্রাহককে কী করতে হবে তা বলা হয়েছে"),
 "phone_number": ("অচেনা নম্বরে যোগাযোগ করতে বলা ঝুঁকিপূর্ণ", "যোগাযোগের নম্বর দেওয়া হয়েছে"),
 "ussd_code": ("ডায়াল কোডটি এখানে সন্দেহজনক প্রেক্ষাপটে ব্যবহৃত হয়েছে",
               "অফারটি নেওয়ার স্বাভাবিক ডায়াল কোড"),
 "money_amount": ("টাকার অঙ্ক দেখিয়ে দ্রুত সাড়া দিতে প্ররোচিত করা হয়েছে",
                  "প্যাকেজের দাম জানানো হয়েছে"),
 "promo_offer": ("অফারের ভাষা ব্যবহার করে বার্তাটিকে বৈধ দেখানোর চেষ্টা হয়েছে",
                 "প্যাকেজ বা অফারের স্বাভাবিক তথ্য"),
 "linguistic_indicator": ("এমন শব্দচয়ন প্রতারণামূলক বার্তায় বেশি দেখা যায়",
                          "এই শব্দগুলো বার্তাটির ধরন বুঝতে সাহায্য করেছে"),
}
def why(t, label):
    a, b = WHY.get(t, WHY["linguistic_indicator"])
    return a if label == "SPAM" else b

ACTION = {
 "SPAM": ["কোনো লিংকে ক্লিক করবেন না এবং পিন, ওটিপি বা পাসওয়ার্ড কাউকে দেবেন না।",
          "বার্তাটির নির্দেশ অনুসরণ করবেন না; প্রয়োজনে প্রতিষ্ঠানের অফিসিয়াল নম্বরে যোগাযোগ করুন।"],
 "PROMO":["অফারটি নিতে চাইলে অপারেটরের অফিসিয়াল অ্যাপ বা কাস্টমার কেয়ারে যাচাই করে নিন।",
          "আগ্রহ না থাকলে বার্তাটি উপেক্ষা করতে পারেন; ব্যক্তিগত তথ্য দেওয়ার প্রয়োজন নেই।"],
 "NORMAL":["এখানে বিশেষ ঝুঁকির লক্ষণ পাওয়া যায়নি, তবে সবসময় সতর্ক থাকা ভালো।",
           "বার্তাটি স্বাভাবিক মনে হচ্ছে; আলাদা কোনো পদক্ষেপ নেওয়ার প্রয়োজন নেই।"]}

DEPTH = {  # HIGH's extra analytical sentence — about the MESSAGE, never about the model
 "SPAM": ["একাধিক সন্দেহজনক উপাদান একসঙ্গে থাকাই এই বার্তাটিকে বেশি ঝুঁকিপূর্ণ করে তুলেছে।",
          "আলাদাভাবে দেখলে উপাদানগুলো নিরীহ মনে হতে পারে, কিন্তু একসঙ্গে এগুলো পরিচিত প্রতারণার ধরন তৈরি করে।"],
 "PROMO":["এই উপাদানগুলো বৈধ প্রচারমূলক বার্তার স্বাভাবিক বৈশিষ্ট্য, তাই এগুলো ঝুঁকির লক্ষণ হিসেবে ধরা হয়নি।",
          "দাম, মেয়াদ বা ডায়াল কোড থাকা মানেই বার্তাটি প্রতারণামূলক নয়।"],
 "NORMAL":["উপলব্ধ প্রমাণে সন্দেহজনক কোনো শক্ত লক্ষণ পাওয়া যায়নি।",
           "বার্তাটির ভাষা ও গঠন সাধারণ তথ্যমূলক বার্তার মতোই।"]}

def _pick(seq, key):
    return seq[int(hashlib.md5(key.encode("utf-8")).hexdigest()[:8], 16) % len(seq)]

def _q(s):
    return "\u201c" + str(s).strip() + "\u201d"

def _ev_list(ev, k):
    return "; ".join(f"{_q(e['text'])} ({e['type_bn']})" for e in ev[:k])

In [11]:
def _assemble(body, action, max_sents):
    """Truncate by ACTUAL sentence count, not by item count — a quoted evidence span can
    still contribute more than one sentence. The safety advice is always last."""
    body = [b for b in body if b and b.strip()]
    budget = max_sents - sent_count(action)
    out, used = [], 0
    for b in body:
        c = sent_count(b)
        if out and used + c > budget:
            break
        out.append(b); used += c
    return " ".join(out + [action])

def _build_target(sms, pred_label, confidence, evidence, level):
    """Deterministic, evidence-grounded Bangla target.
    ALWAYS explains `pred_label`. Contains NO generated numbers."""
    L = pred_label if pred_label in LABEL_BN else "NORMAL"
    key = sms + level
    op = _pick(OPENERS[level][L], key)
    act = _pick(ACTION[L], key)
    ev = evidence or []

    # ---------------- LOW: 1-2 sentences -------------------------------------
    if level == "low":
        if not ev:
            return f"{op} {act}"
        cue = (f"এখানে {_q(ev[0]['text'])} ও {_q(ev[1]['text'])} অংশ দুটি সবচেয়ে গুরুত্বপূর্ণ"
               if len(ev) > 1 else
               f"এখানে {_q(ev[0]['text'])} অংশটি সবচেয়ে গুরুত্বপূর্ণ")
        # merged into the opener with a semicolon so LOW stays at two sentences
        return f"{op.rstrip('।')}; {cue}। {act}"

    # ---------------- MEDIUM: 2-4 sentences ----------------------------------
    if level == "medium":
        if not ev:
            body = [op, "বার্তাটির নির্দিষ্ট কোনো অংশ আলাদা করে শক্ত প্রমাণ হিসেবে পাওয়া "
                        "যায়নি; সিদ্ধান্তটি সামগ্রিক ভাষার ধরন থেকে এসেছে।"]
        else:
            body = [op,
                    f"সিদ্ধান্তে সবচেয়ে বেশি প্রভাব ফেলেছে: {_ev_list(ev, 2)}।",
                    f"এটি গুরুত্বপূর্ণ কারণ {why(ev[0]['type'], L)}।"]
        return _assemble(body, act, 4)

    # ---------------- HIGH: 4-7 sentences, deeper, no jargon -----------------
    if level == "high":
        body = [op, f"{CONF_BN(confidence)}।"]
        if not ev:
            body.append("বার্তাটির কোনো একক অংশ নির্ণায়ক প্রমাণ হিসেবে উঠে আসেনি, তাই "
                        "সিদ্ধান্তটি বার্তার সামগ্রিক ভাষা ও গঠনের উপর নির্ভরশীল।")
        else:
            body.append(f"বার্তার যে অংশগুলো সিদ্ধান্তে সবচেয়ে বেশি ভূমিকা রেখেছে: {_ev_list(ev, 3)}।")
            body.append(f"এর মধ্যে {_q(ev[0]['text'])} সবচেয়ে গুরুত্বপূর্ণ, কারণ {why(ev[0]['type'], L)}।")
            if len(ev) > 1:
                body.append(f"পাশাপাশি {_q(ev[1]['text'])} অংশটিও প্রাসঙ্গিক, কারণ "
                            f"{why(ev[1]['type'], L)}।")
        body.append(_pick(DEPTH[L], key))
        if confidence < 0.70:
            body.append("যেহেতু সিদ্ধান্তটি সম্পূর্ণ নিশ্চিত নয়, তাই নিজে যাচাই করে নেওয়া ভালো।")
        return _assemble(body, act, 7)

    # ---------------- STATIC: 2-5 sentences ----------------------------------
    if ev:
        body = [op, f"বার্তাটিতে উল্লেখযোগ্য অংশ: {_ev_list(ev, 2)}।",
                f"এটি প্রাসঙ্গিক কারণ {why(ev[0]['type'], L)}।"]
    else:
        body = [op, "বার্তাটির নির্দিষ্ট কোনো অংশ প্রধান প্রমাণ হিসেবে চিহ্নিত হয়নি।"]
    return _assemble(body, act, 5)


SENT_LIMITS = {"low": (1, 2), "medium": (2, 4), "high": (4, 7), "static": (2, 5)}

def make_target(sms, pred_label, confidence, evidence, level):
    """Wraps _build_target with a hard guarantee: the returned string is always inside its
    sentence budget and always free of malformed/garbage patterns. If quoting many
    evidence spans trips a check (e.g. four numeric spans in a row look like a numeric
    run), progressively fewer cues are used until the output is clean."""
    ev = list(evidence or [])
    lo, hi = SENT_LIMITS[level]
    best = None
    for k in range(len(ev), -1, -1):
        t = _build_target(sms, pred_label, confidence, ev[:k], level)
        n = sent_count(t)
        if best is None:
            best = t
        if lo <= n <= hi and not garbage_codes(t):
            return t
    t = _build_target(sms, pred_label, confidence, [], level)
    return t if (lo <= sent_count(t) <= hi and not garbage_codes(t)) else best


# ---- regenerate every target from the PREDICTED label ----------------------
NEW_COL = {lv: f"new_{lv}" for lv in LEVELS}
for lv in LEVELS:
    df[NEW_COL[lv]] = [make_target(r["SMS"], r["predicted_label"], r["confidence"],
                                   r["evidence"], lv) for _, r in df.iterrows()]


print("Regenerated targets:")
for lv in LEVELS:
    sc = df[NEW_COL[lv]].map(sent_count); lo, hi = SENT_LIMITS[lv]
    print(f"  {lv:7s} sents med {sc.median():.0f} range {sc.min():.0f}-{sc.max():.0f} "
          f"(allowed {lo}-{hi})  chars med {df[NEW_COL[lv]].str.len().median():.0f}  "
          f"max digit-ratio {df[NEW_COL[lv]].map(digit_ratio).max():.3f}")

bad_contra = sum(1 for _, r in df.iterrows() for lv in LEVELS
                 if (detect_asserted_class(r[NEW_COL[lv]]) or r["predicted_label"]) != r["predicted_label"])
bad_garb = sum(1 for _, r in df.iterrows() for lv in LEVELS if garbage_codes(r[NEW_COL[lv]]))
bad_len = sum(1 for _, r in df.iterrows() for lv in LEVELS
              if not (SENT_LIMITS[lv][0] <= sent_count(r[NEW_COL[lv]]) <= SENT_LIMITS[lv][1]))
print(f"\ncontradictions in NEW targets : {bad_contra}")
print(f"garbage codes in NEW targets  : {bad_garb}")
print(f"length violations             : {bad_len}")
assert bad_contra == 0 and bad_garb == 0 and bad_len == 0, "new targets must be clean"

print("\nDistinct openers per level now:")
for lv in LEVELS:
    print(f"  {lv:7s}: {df[NEW_COL[lv]].map(lambda t: _SENT_SPLIT.split(t)[0][:40]).nunique()}")

print("\n--- sample (same SMS, all four levels) ---")
_r = df[df["predicted_label"] == "SPAM"].iloc[0] if (df["predicted_label"] == "SPAM").any() else df.iloc[0]
print("SMS:", _r["SMS"][:160])
for lv in LEVELS:
    print(f"\n[{lv.upper()}] {_r[NEW_COL[lv]]}")

Regenerated targets:
  static  sents med 4 range 3-5 (allowed 2-5)  chars med 277  max digit-ratio 0.078
  low     sents med 2 range 2-2 (allowed 1-2)  chars med 164  max digit-ratio 0.117
  medium  sents med 4 range 2-4 (allowed 2-4)  chars med 297  max digit-ratio 0.076
  high    sents med 7 range 5-7 (allowed 4-7)  chars med 544  max digit-ratio 0.089

contradictions in NEW targets : 0
garbage codes in NEW targets  : 0
length violations             : 0

Distinct openers per level now:
  static : 3
  low    : 455
  medium : 6
  high   : 6

--- sample (same SMS, all four levels) ---
SMS: 🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁 ⭕️!!️ 🔠🔠🔠 🟣⚪️🟣⚪️🟣🟣⭐⭐ ⏰ কোড: NG88TS7 📌 আগে আসলে আগে পাবেন - সীমিত ভাউচার সংখ্যা! ⏳কোডটি শেষ হয়ে যাওয়ার আগে ব্যবহার করুন! চোখ রাখুন টেলিগ্রামে 🎯 য

[STATIC] শ্রেণিবিন্যাস অনুযায়ী এটি একটি প্রতারণামূলক (স্মিশিং) বার্তা। বার্তাটিতে উল্লেখযোগ্য অংশ: “কোড:” (গোপন তথ্য চাওয়া); “supporting,” (পরিচয় ভাঁড়ানোর ইঙ্গিত)। এটি প্রাসঙ্গিক কারণ পিন, ওটিপি বা পাসওয়ার্ড কোনো প্রতিষ্ঠান এসএমএসে চায় 

## 7. Output validator and deterministic fallback

Every generated string passes through this before it can be shown. Any failure routes to
the evidence-grounded realiser, so an invalid output can never reach a user.

```
BanglaT5 output → validator → PASS → shown
                            → FAIL → make_target(...) fallback → shown
```

In [12]:
BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
NUMTOK_RE = re.compile(r"\d[\d,\.]*")
_TRIM = "\u201c\u201d\"'()[]{}<>।,;:!? "
ORG_HINTS = ["বিকাশ", "নগদ", "রকেট", "গ্রামীণফোন", "রবি", "এয়ারটেল", "বাংলালিংক", "টেলিটক",
             "ব্র্যাক", "সোনালী", "ইসলামী", "bkash", "nagad", "rocket", "grameenphone",
             "robi", "airtel", "banglalink", "teletalk", "paypal", "amazon", "daraz"]
JARGON = ["গ্রেডিয়েন্ট", "লজিট", "এমবেডিং", "হিডেন স্টেট", "টোকেন-পর্যায়",
          "ইন্টিগ্রেটেড গ্রেডিয়েন্ট", "gradient", "logit", "embedding", "hidden state"]

def _canon(x):
    x = str(x).replace(",", "")
    return x.rstrip("0").rstrip(".") if "." in x else x

def validate_output(text, sms, pred_label, evidence, level):
    """Returns (ok, [failure codes]). Ordered so the most serious appear first."""
    t = str(text).strip()
    f = []

    # --- 1. classifier contradiction (the headline requirement) -------------
    asserted = detect_asserted_class(t)
    if asserted is not None and asserted != pred_label:
        f.append(f"classifier_contradiction:{asserted}!={pred_label}")

    # --- 2. malformed / garbage --------------------------------------------
    f += [f"malformed:{c}" for c in garbage_codes(t)]

    # --- 3. length ----------------------------------------------------------
    lo, hi = SENT_LIMITS[level]
    n = sent_count(t)
    if n < lo: f.append(f"too_few_sentences:{n}<{lo}")
    if n > hi: f.append(f"too_many_sentences:{n}>{hi}")
    if len(t) < 20: f.append("too_short")
    if len(t) > 1400: f.append("too_long")

    # --- 4. unsupported claims (nothing may be invented) --------------------
    for u in URL_RE.findall(t):
        if u.strip().strip(_TRIM) not in sms:
            f.append(f"unsupported_url:{u[:32]}")
    for ph in PHONE_RE.findall(t):
        if ph.strip(_TRIM) not in sms:
            f.append(f"unsupported_phone:{ph}")
    allowed = {_canon(x) for x in NUMTOK_RE.findall(str(sms).translate(BN_DIGITS))}
    for e in (evidence or []):
        allowed |= {_canon(x) for x in NUMTOK_RE.findall(str(e["text"]).translate(BN_DIGITS))}
    for num in NUMTOK_RE.findall(t.translate(BN_DIGITS)):
        if _canon(num) not in allowed:
            f.append(f"unsupported_number:{num}")
    low_t, low_s = t.lower(), str(sms).lower()
    for org in ORG_HINTS:
        if org in low_t and org not in low_s:
            f.append(f"unsupported_org:{org}")

    # --- 5. evidence contradiction: don't claim a cue the SMS lacks ---------
    ev_types = {e["type"] for e in (evidence or [])}
    if ("লিংক" in t or "URL" in t.upper()) and not (URL_RE.search(sms) or
            ev_types & {"url", "suspicious_url"}) and pred_label != "SPAM":
        f.append("evidence_contradiction:claims_link")
    if ("ফোন নম্বর" in t) and not (PHONE_RE.search(sms) or "phone_number" in ev_types):
        f.append("evidence_contradiction:claims_phone")

    # --- 6. quoted spans must exist in the SMS ------------------------------
    for q in re.findall(r"[\u201c\u201d]([^\u201c\u201d]{1,80})[\u201c\u201d]", t):
        if q.strip() and q.strip() not in sms:
            f.append(f"quote_not_in_sms:{q[:24]}")

    # --- 7. ML jargon (HIGH must not expose internals) ----------------------
    for j in JARGON:
        if j.lower() in low_t:
            f.append(f"ml_jargon:{j}")

    return (len(f) == 0), f

def explain_with_fallback(raw, sms, pred_label, confidence, evidence, level):
    """Never returns an invalid explanation."""
    ok, fails = validate_output(raw, sms, pred_label, evidence, level)
    if ok:
        return raw, "model", []
    fb = make_target(sms, pred_label, confidence, evidence, level)
    ok2, f2 = validate_output(fb, sms, pred_label, evidence, level)
    return fb, ("fallback" if ok2 else "fallback_unvalidated"), fails

# ---- validator self-test on known-bad strings ------------------------------
_sms = "আজই ২৫টাকায় ৫জিবি নিতে ডায়াল *21291*825#"
_ev = [{"text": "*21291*825#", "type": "ussd_code", "type_bn": "ডায়াল কোড", "importance": .5}]
_bad = {
 "garbage numbers":      "173:37298:1993 97:97:98 এটি একটি প্রচারমূলক বার্তা। যাচাই করে নিন।",
 "classifier contradiction": "এই বার্তাটি প্রতারণামূলক (স্মিশিং) বলে শনাক্ত হয়েছে। সতর্ক থাকুন।",
 "invented url":         "এটি প্রচারমূলক বার্তা। https://evil.com/x দেখুন। যাচাই করে নিন।",
 "ml jargon":            "এটি প্রচারমূলক বার্তা। ইন্টিগ্রেটেড গ্রেডিয়েন্ট বিশ্লেষণে দেখা যায়। যাচাই করুন।",
 "prompt leak":          "SMS: আজই ২৫টাকায় LABEL: PROMO এটি প্রচারমূলক বার্তা। যাচাই করুন।",
 "empty":                "",
}
print("validator self-test (all must be caught):")
for name, s in _bad.items():
    ok, f = validate_output(s, _sms, "PROMO", _ev, "medium")
    print(f"  {'CAUGHT ' if not ok else 'MISSED!'} {name:26s} {f[:2]}")
    assert not ok, f"validator missed: {name}"

_good = make_target(_sms, "PROMO", 0.95, _ev, "medium")
ok, f = validate_output(_good, _sms, "PROMO", _ev, "medium")
print(f"\n  {'PASS' if ok else 'FAIL'} clean grounded target -> {f}")
assert ok

validator self-test (all must be caught):
  CAUGHT  garbage numbers            ['malformed:colon_number_garbage', 'malformed:numeric_run']
  CAUGHT  classifier contradiction   ['classifier_contradiction:SPAM!=PROMO']
  CAUGHT  invented url               ['unsupported_url:https://evil.com/x']
  CAUGHT  ml jargon                  ['ml_jargon:গ্রেডিয়েন্ট', 'ml_jargon:ইন্টিগ্রেটেড গ্রেডিয়েন্ট']
  CAUGHT  prompt leak                ['malformed:prompt_leak']
  CAUGHT  empty                      ['malformed:empty', 'too_few_sentences:0<2']

  PASS clean grounded target -> []


## 8. Training data — inputs carry the PREDICTED label

`source_split` is used exactly as it is; nothing is reshuffled and the test set is never
seen during training. The critical line is `LABEL: {predicted_label}` — the previous
notebook wrote the gold label here while the target explained the prediction, which is
what taught the model to ignore the field.

In [13]:
def build_input(sms, pred_label, evidence, level):
    ev = "; ".join(f"{e['text']} ({e['type_bn']})" for e in (evidence or [])[:4]) or "নেই"
    return (f"SMS: {sms}\nLABEL: {pred_label}\nXAI EVIDENCE: {ev}\n"
            f"EXPLANATION TYPE: {level.upper()}")

df["_split"] = df["source_split"].str.lower().replace(
    {"val": "validation", "valid": "validation", "dev": "validation"})
print("splits:\n", df["_split"].value_counts().to_string())
if "validation" not in set(df["_split"]):
    raise ValueError("No validation split in source_split — cannot early-stop safely.")

def level_frame(level, split):
    sub = df[df["_split"] == split]
    return pd.DataFrame({
        "input_text": [build_input(r["SMS"], r["predicted_label"], r["evidence"], level)
                       for _, r in sub.iterrows()],
        "target_text": sub[NEW_COL[level]].tolist(),
        "SMS": sub["SMS"].tolist(), "gold": sub["label"].tolist(),
        "pred": sub["predicted_label"].tolist(),
        "confidence": sub["confidence"].tolist(),
        "evidence": sub["evidence"].tolist(),
        "old_expl": sub[LEVEL_COL[level]].tolist(),
    }).reset_index(drop=True)

for lv in LEVELS:
    print(f"  {lv:7s} train={len(level_frame(lv,'train'))} "
          f"val={len(level_frame(lv,'validation'))} test={len(level_frame(lv,'test'))}")

print("\nexample input (note LABEL is the PREDICTED label):")
print(level_frame("high", "train")["input_text"].iloc[0])
print("\nexample target:")
print(level_frame("high", "train")["target_text"].iloc[0])

splits:
 _split
train         1319
validation     170
test           152
  static  train=1319 val=170 test=152
  low     train=1319 val=170 test=152
  medium  train=1319 val=170 test=152
  high    train=1319 val=170 test=152

example input (note LABEL is the PREDICTED label):
SMS: 🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁🎁 ⭕️!!️ 🔠🔠🔠 🟣⚪️🟣⚪️🟣🟣⭐⭐ ⏰ কোড: NG88TS7 📌 আগে আসলে আগে পাবেন - সীমিত ভাউচার সংখ্যা! ⏳কোডটি শেষ হয়ে যাওয়ার আগে ব্যবহার করুন! চোখ রাখুন টেলিগ্রামে 🎯 যেকোনো সময়ে পরের ড্রপ হতে পারে 💬 NAGAD88 সবসময়ই আলাদা! ❤️ আপনাদের ভালোবাসার জন্যই আজ আমরা দিচ্ছি এক্সক্লুসিভ ভাউচার ড্রপ 🎟️ কারণ আমরা বিশ্বাস করি - যারা নিয়মিত থাকে, তারাই পায় সবচেয়ে বড় সারপ্রাইজ! 🎉 🌟 Keep supporting, keep enjoying. ➡️➡️শর্তাবলী প্রযোজ্য (Terms & Conditions Apply) 💎 Play Safe, Play NAGAD88 ⭐সফ মানে কী? 💵 টাকা উত্তোলনের গ্যারান্টি ⚡️ দ্রুত টাকা উত্তোলন আরও বড় খবর আসছে শীঘ্রই! 🔥 #PlaySafe #PlayNAGAD88🧬
LABEL: SPAM
XAI EVIDENCE: কোড: (গোপন তথ্য চাওয়া); supporting, (পরিচয় ভাঁড়ানোর ইঙ্গিত); শীঘ্রই! (তাড়াহুড়ার ভাষা); এক্সক্লুসিভ (

In [14]:
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback)
from datasets import Dataset as HFDataset
import inspect as _insp

_ARGS = set(_insp.signature(Seq2SeqTrainingArguments.__init__).parameters)
_TR   = set(_insp.signature(Seq2SeqTrainer.__init__).parameters)

def load_t5_tokenizer():
    try:                     # csebuetnlp recommends the sentencepiece (slow) tokenizer
        return AutoTokenizer.from_pretrained(BANGLAT5_ID, use_fast=False)
    except Exception as e:
        print("slow tokenizer unavailable, using fast:", e)
        return AutoTokenizer.from_pretrained(BANGLAT5_ID)

def to_ds(frame, tok, max_target):
    ds = HFDataset.from_pandas(frame[["input_text", "target_text"]], preserve_index=False)
    def _t(b):
        enc = tok(b["input_text"], truncation=True, max_length=MAX_SOURCE)
        enc["labels"] = tok(text_target=b["target_text"], truncation=True,
                            max_length=max_target)["input_ids"]
        return enc
    return ds.map(_t, batched=True, remove_columns=["input_text", "target_text"])

def train_level(level):
    cfg = LEVEL_CFG[level]
    print("\n" + "=" * 74); print(f"TRAINING {level.upper()}  {cfg}"); print("=" * 74)
    set_seed(SEED)
    tok = load_t5_tokenizer()
    model = AutoModelForSeq2SeqLM.from_pretrained(BANGLAT5_ID).to(DEVICE)

    tr = to_ds(level_frame(level, "train"), tok, cfg["max_target"])
    va = to_ds(level_frame(level, "validation"), tok, cfg["max_target"])

    kw = dict(output_dir=str(OUT / "models" / f"ckpt_{level}"),
              num_train_epochs=cfg["epochs"],
              per_device_train_batch_size=cfg["batch"],
              per_device_eval_batch_size=max(8, cfg["batch"]),
              gradient_accumulation_steps=cfg["accum"],
              learning_rate=cfg["lr"], weight_decay=0.01, warmup_ratio=0.06,
              lr_scheduler_type="linear", max_grad_norm=1.0,
              logging_steps=50, seed=SEED, data_seed=SEED, report_to="none",
              save_total_limit=1, load_best_model_at_end=True,
              metric_for_best_model="eval_loss", greater_is_better=False,
              predict_with_generate=False,
              bf16=(DEVICE == "cuda" and BF16), fp16=False,   # T5 is unstable in fp16
              dataloader_num_workers=0)
    if "eval_strategy" in _ARGS:        kw["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in _ARGS: kw["evaluation_strategy"] = "epoch"
    kw["save_strategy"] = "epoch"
    args = Seq2SeqTrainingArguments(**{k: v for k, v in kw.items() if k in _ARGS})

    tkw = dict(model=model, args=args, train_dataset=tr, eval_dataset=va,
               data_collator=DataCollatorForSeq2Seq(tok, model=model),
               callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    tkw["processing_class" if "processing_class" in _TR else "tokenizer"] = tok

    trainer = Seq2SeqTrainer(**tkw)
    t0 = time.time(); trainer.train(); el = time.time() - t0
    vloss = float(trainer.evaluate()["eval_loss"])

    folder = "static_explanation" if level == "static" else f"adaptive_{level}"
    d = DEPLOY / folder
    (d / "model").mkdir(parents=True, exist_ok=True)
    (d / "tokenizer").mkdir(parents=True, exist_ok=True)
    model.save_pretrained(d / "model"); tok.save_pretrained(d / "tokenizer")
    json.dump({"base_model": BANGLAT5_ID, "level": level, **{k: v for k, v in cfg.items()},
               "max_source": MAX_SOURCE, "seed": SEED, "val_loss": vloss,
               "train_seconds": el, "n_train": len(tr), "n_val": len(va),
               "input_label_field": "predicted_label (classifier is source of truth)"},
              open(d / "training_config.json", "w", encoding="utf-8"),
              indent=2, ensure_ascii=False)
    print(f"saved -> {d} | val_loss {vloss:.4f} | {el:.0f}s")
    del trainer, model; free_gpu()
    return {"level": level, "val_loss": vloss, "seconds": el}

TRAIN_INFO = []
if RUN_TRAIN:
    for lv in LEVELS:
        TRAIN_INFO.append(train_level(lv))
    print("\n", pd.DataFrame(TRAIN_INFO).to_string(index=False))
    json.dump(TRAIN_INFO, open(OUT / "results" / "training_info.json", "w",
                               encoding="utf-8"), indent=2)
else:
    print("training skipped (RUN_TRAIN=False) — existing checkpoints will be reloaded.")


TRAINING STATIC  {'max_target': 144, 'epochs': 8, 'lr': 0.0003, 'batch': 8, 'accum': 2, 'gen': {'min_new_tokens': 25, 'max_new_tokens': 140, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'repetition_penalty': 1.2, 'length_penalty': 1.0}}


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/170 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Training Loss,Validation Loss,Epoch
0.341262,0.047000,12


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved -> deployment\adaptive_high | val_loss 0.0470 | 482s

  level  val_loss    seconds
static  0.031663 218.146930
   low  0.042332 213.126932
medium  0.037605 228.927287
  high  0.047000 481.532422


## 9. Generation, validation and evaluation on the held-out TEST split

Deterministic beam search with per-level parameters. Every output goes through the
validator; failures fall back. Reported per level: ROUGE-L, validity rate, classifier
consistency, contradiction rate, evidence consistency, unsupported-claim rate,
malformed rate, fallback rate and mean length.

In [15]:
def _tok_words(s):
    return [w for w in re.split(r"\s+", str(s).strip()) if w]

def _lcs(a, b):
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            dp[i][j] = dp[i-1][j-1]+1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[-1][-1]

def rouge_l(pred, ref):
    """Whitespace-token ROUGE-L. An English tokenizer would mangle Bangla."""
    a, b = _tok_words(pred), _tok_words(ref)
    if not a or not b: return 0.0
    l = _lcs(a, b)
    if l == 0: return 0.0
    p, r = l/len(a), l/len(b)
    return 2*p*r/(p+r)

@torch.no_grad()
def generate_batch(model, tok, inputs, gen_cfg, batch_size=8):
    outs = []
    for i in range(0, len(inputs), batch_size):
        enc = tok(list(inputs[i:i+batch_size]), return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_SOURCE).to(DEVICE)
        g = model.generate(**enc, do_sample=False, early_stopping=True, **gen_cfg)
        outs += tok.batch_decode(g, skip_special_tokens=True)
    return outs

def evaluate_level(level, limit=None):
    folder = "static_explanation" if level == "static" else f"adaptive_{level}"
    d = DEPLOY / folder
    if not (d / "model").exists():
        print(f"[{level}] no checkpoint — skipped"); return None, None
    tok = AutoTokenizer.from_pretrained(d / "tokenizer")
    model = AutoModelForSeq2SeqLM.from_pretrained(d / "model").to(DEVICE).eval()

    te = level_frame(level, "test")
    if limit: te = te.head(limit)
    raws = generate_batch(model, tok, te["input_text"].tolist(), LEVEL_CFG[level]["gen"])

    rows = []
    for k in range(len(te)):
        sms, pred = te["SMS"].iloc[k], te["pred"].iloc[k]
        ev, conf = te["evidence"].iloc[k], float(te["confidence"].iloc[k])
        raw = raws[k]
        ok, fails = validate_output(raw, sms, pred, ev, level)
        final, source, _ = explain_with_fallback(raw, sms, pred, conf, ev, level)
        asserted_raw   = detect_asserted_class(raw)
        asserted_final = detect_asserted_class(final)
        rows.append({
            "level": level, "SMS": sms, "gold": te["gold"].iloc[k], "pred": pred,
            "confidence": conf, "raw": raw, "final": final, "source": source,
            "reference": te["target_text"].iloc[k], "old_expl": te["old_expl"].iloc[k],
            "valid_raw": ok, "fail_codes": ";".join(fails[:4]),
            "rougeL_raw": rouge_l(raw, te["target_text"].iloc[k]),
            "rougeL_final": rouge_l(final, te["target_text"].iloc[k]),
            "contradiction_raw":   bool(asserted_raw is not None and asserted_raw != pred),
            "contradiction_final": bool(asserted_final is not None and asserted_final != pred),
            "malformed_raw": bool(garbage_codes(raw)),
            "unsupported_raw": any(f.startswith("unsupported") for f in fails),
            "evidence_ok": not any(f.startswith(("evidence_contradiction", "quote_not_in_sms"))
                                   for f in fails),
            "n_evidence": len(ev),
            "evidence_str": "; ".join(f"{e['text']} ({e['type_bn']})" for e in ev) or "নেই",
            "words_final": len(_tok_words(final)),
            "sents_final": sent_count(final),
        })
    del model; free_gpu()
    r = pd.DataFrame(rows)
    summary = {
        "n_test": int(len(r)),
        "rougeL": float(r["rougeL_final"].mean()),
        "rougeL_raw_model_only": float(r["rougeL_raw"].mean()),
        "validity_rate": float(r["valid_raw"].mean()),
        "classifier_consistency_rate": float(1 - r["contradiction_final"].mean()),
        "contradiction_rate": float(r["contradiction_final"].mean()),
        "contradiction_rate_before_fallback": float(r["contradiction_raw"].mean()),
        "evidence_consistency_rate": float(r["evidence_ok"].mean()),
        "unsupported_claim_rate": float(r["unsupported_raw"].mean()),
        "malformed_rate": float(r["malformed_raw"].mean()),
        "fallback_rate": float((r["source"] != "model").mean()),
        "avg_length_words": float(r["words_final"].mean()),
        "avg_sentences": float(r["sents_final"].mean()),
    }
    return r, summary

EVAL, GEN_ROWS = {}, []
if RUN_EVAL:
    lim = 40 if QUICK_TEST else None
    for lv in LEVELS:
        r, s = evaluate_level(lv, limit=lim)
        if r is None: continue
        EVAL[lv] = s; GEN_ROWS.append(r)
        print(f"\n[{lv.upper()}]"); print(json.dumps(s, indent=2))
    if GEN_ROWS:
        gen_df = pd.concat(GEN_ROWS, ignore_index=True)
        gen_df.to_csv(OUT / "results" / "test_generations.csv", index=False, encoding="utf-8-sig")
        eval_df = pd.DataFrame(EVAL).T; eval_df.index.name = "level"
        print("\n" + "=" * 74); print("NEW SYSTEM — test set"); print("=" * 74)
        print(eval_df.to_string())
        eval_df.to_csv(OUT / "results" / "evaluation_new.csv")
        json.dump(EVAL, open(OUT / "results" / "evaluation_new.json", "w",
                             encoding="utf-8"), indent=2, ensure_ascii=False)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



[STATIC]
{
  "n_test": 152,
  "rougeL": 0.772215348084709,
  "rougeL_raw_model_only": 0.765644273209699,
  "validity_rate": 0.9605263157894737,
  "classifier_consistency_rate": 1.0,
  "contradiction_rate": 0.0,
  "contradiction_rate_before_fallback": 0.0,
  "evidence_consistency_rate": 1.0,
  "unsupported_claim_rate": 0.03289473684210526,
  "malformed_rate": 0.006578947368421052,
  "fallback_rate": 0.039473684210526314,
  "avg_length_words": 37.57236842105263,
  "avg_sentences": 4.144736842105263
}


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



[MEDIUM]
{
  "n_test": 152,
  "rougeL": 0.7507610496643423,
  "rougeL_raw_model_only": 0.7266152881697753,
  "validity_rate": 0.9407894736842105,
  "classifier_consistency_rate": 1.0,
  "contradiction_rate": 0.0,
  "contradiction_rate_before_fallback": 0.0,
  "evidence_consistency_rate": 1.0,
  "unsupported_claim_rate": 0.03289473684210526,
  "malformed_rate": 0.006578947368421052,
  "fallback_rate": 0.05921052631578947,
  "avg_length_words": 40.703947368421055,
  "avg_sentences": 3.986842105263158
}


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



[HIGH]
{
  "n_test": 152,
  "rougeL": 0.7843382858955007,
  "rougeL_raw_model_only": 0.6209183343921596,
  "validity_rate": 0.6118421052631579,
  "classifier_consistency_rate": 1.0,
  "contradiction_rate": 0.0,
  "contradiction_rate_before_fallback": 0.2236842105263158,
  "evidence_consistency_rate": 0.9013157894736842,
  "unsupported_claim_rate": 0.15789473684210525,
  "malformed_rate": 0.006578947368421052,
  "fallback_rate": 0.3881578947368421,
  "avg_length_words": 69.4342105263158,
  "avg_sentences": 6.519736842105263
}

NEW SYSTEM — test set
        n_test    rougeL  rougeL_raw_model_only  validity_rate  classifier_consistency_rate  contradiction_rate  contradiction_rate_before_fallback  evidence_consistency_rate  unsupported_claim_rate  malformed_rate  fallback_rate  avg_length_words  avg_sentences
level                                                                                                                                                                                 

## 10. Old vs new comparison, and the specific problem cases

In [16]:
COMPARE = {}
if GEN_ROWS:
    rows = []
    for lv in LEVELS:
        if lv not in EVAL: continue
        sub = gen_df[gen_df["level"] == lv]
        old_contra = [bool((a := detect_asserted_class(o)) is not None and a != p)
                      for o, p in zip(sub["old_expl"], sub["pred"])]
        old_garb = [bool(garbage_codes(o)) for o in sub["old_expl"]]
        old_dig  = [digit_ratio(o) for o in sub["old_expl"]]
        rows.append({
            "level": lv,
            "contradiction_OLD": float(np.mean(old_contra)),
            "contradiction_NEW": EVAL[lv]["contradiction_rate"],
            "malformed_OLD": float(np.mean(old_garb)),
            "malformed_NEW": EVAL[lv]["malformed_rate"],
            "digit_ratio_OLD": float(np.mean(old_dig)),
            "digit_ratio_NEW": float(np.mean([digit_ratio(t) for t in sub["final"]])),
            "avg_words_OLD": float(np.mean([len(_tok_words(o)) for o in sub["old_expl"]])),
            "avg_words_NEW": EVAL[lv]["avg_length_words"],
        })
    cmp_df = pd.DataFrame(rows).set_index("level")
    print("=" * 78); print("OLD vs NEW (same test messages)"); print("=" * 78)
    print(cmp_df.round(4).to_string())
    cmp_df.to_csv(OUT / "results" / "old_vs_new.csv")
    COMPARE = cmp_df.to_dict()

    print("\n" + "=" * 78)
    print("SPECIFIC PROBLEM CASES — old behaviour vs new")
    print("=" * 78)

    def show(title, mask, n=2, level=None):
        sub = gen_df[mask] if level is None else gen_df[mask & (gen_df["level"] == level)]
        print(f"\n### {title}  (found {len(sub)})")
        for _, r in sub.head(n).iterrows():
            print("-" * 70)
            print("SMS       :", r["SMS"][:150])
            print(f"CLASSIFIER: {r['pred']}  (gold {r['gold']}, conf {r['confidence']:.3f})")
            print(f"OLD [{r['level']}]:", str(r["old_expl"])[:230])
            print(f"NEW [{r['level']}]:", str(r["final"])[:230])
            print(f"           old asserted={detect_asserted_class(r['old_expl'])} -> "
                  f"new asserted={detect_asserted_class(r['final'])}")

    old_ass = gen_df["old_expl"].map(detect_asserted_class)
    show("Classifier NORMAL but OLD explanation said SPAM",
         (gen_df["pred"] == "NORMAL") & (old_ass == "SPAM"))
    show("Classifier SPAM but OLD explanation said NORMAL",
         (gen_df["pred"] == "SPAM") & (old_ass == "NORMAL"))
    show("Classifier PROMO but OLD explanation said SPAM",
         (gen_df["pred"] == "PROMO") & (old_ass == "SPAM"))
    show("MEDIUM/HIGH numeric garbage in OLD targets",
         gen_df["old_expl"].map(lambda t: bool(garbage_codes(t))) &
         gen_df["level"].isin(["medium", "high"]))
    show("Weak / empty XAI evidence (system must not invent any)",
         gen_df["n_evidence"] == 0)

OLD vs NEW (same test messages)
        contradiction_OLD  contradiction_NEW  malformed_OLD  malformed_NEW  digit_ratio_OLD  digit_ratio_NEW  avg_words_OLD  avg_words_NEW
level                                                                                                                                     
static                0.0                0.0         0.0066         0.0066           0.0188           0.0065        42.0000        37.5724
low                   0.0                0.0         0.0066         0.0066           0.0111           0.0107        27.1645        23.2303
medium                0.0                0.0         0.0066         0.0066           0.0069           0.0062        48.8618        40.7039
high                  0.0                0.0         0.0066         0.0066           0.0304           0.0074        87.2829        69.4342

SPECIFIC PROBLEM CASES — old behaviour vs new

### Classifier NORMAL but OLD explanation said SPAM  (found 0)

### Classifier SPAM bu

## 11. Fifty TEST messages, printed in full

17 NORMAL / 17 PROMO / 16 SPAM by **gold** label, drawn from the test split only. The SMS
is printed complete and unmodified. Any classifier/gold disagreement in the sample is
reported rather than hidden.

In [17]:
def pick_balanced(n_total=50):
    """17 NORMAL / 17 PROMO / 16 SPAM from the TEST split, by gold label."""
    quota = {"NORMAL": 17, "PROMO": 17, "SPAM": 16}
    assert sum(quota.values()) == n_total
    base = gen_df[gen_df["level"] == "high"][["SMS", "gold", "pred", "confidence",
                                             "n_evidence", "evidence_str"]].drop_duplicates("SMS")
    rng = np.random.RandomState(SEED)
    picked = []
    for lab, k in quota.items():
        pool = base[base["gold"] == lab]
        if len(pool) == 0:
            print(f"  WARNING: no TEST messages with gold={lab}")
            continue
        # prefer rows where the classifier agreed, then top up if short
        agree = pool[pool["gold"] == pool["pred"]]
        take = agree if len(agree) >= k else pool
        idx = rng.choice(len(take), min(k, len(take)), replace=False)
        picked.append(take.iloc[idx])
        if len(take) < k:
            print(f"  WARNING: only {len(take)} TEST messages available for {lab} (wanted {k})")
    return pd.concat(picked, ignore_index=True) if picked else pd.DataFrame()

sel = pick_balanced(N_PRINT)
by_level = {lv: gen_df[gen_df["level"] == lv].set_index("SMS")["final"].to_dict()
            for lv in LEVELS if lv in EVAL}
src_level = {lv: gen_df[gen_df["level"] == lv].set_index("SMS")["source"].to_dict()
             for lv in LEVELS if lv in EVAL}

print(f"Printing {len(sel)} TEST messages "
      f"({dict(sel['gold'].value_counts())})\n")
disagree = 0
for i, r in sel.iterrows():
    sms = r["SMS"]
    print("=" * 100)
    print(f"[{i+1}/{len(sel)}]")
    print("\nSMS:")
    print(sms)                                   # full text, never truncated
    print("\nGOLD:")
    print(r["gold"])
    print("\nCLASSIFIER:")
    print(r["pred"] + ("" if r["gold"] == r["pred"] else "   <-- disagrees with gold"))
    if r["gold"] != r["pred"]:
        disagree += 1
    print("\nCONFIDENCE:")
    print(f"{r['confidence']*100:.2f}%")
    print("\nXAI EVIDENCE:")
    print(r["evidence_str"])
    checks = []
    for lv in LEVELS:
        if lv not in by_level:
            continue
        txt = by_level[lv].get(sms, "")
        print(f"\n[{lv.upper()}]")
        print(txt)
        a = detect_asserted_class(txt)
        checks.append((lv, a is None or a == r["pred"], src_level[lv].get(sms, "?")))
    print("\nVALIDATION:")
    allok = all(c[1] for c in checks)
    print(f"  classifier consistency : {'PASS — all four levels explain ' + r['pred'] if allok else 'FAIL'}")
    print("  source per level       : " + ", ".join(f"{lv}={s}" for lv, _, s in checks))
    print("=" * 100 + "\n")

print(f"\nSummary of the {len(sel)} printed messages:")
print(f"  gold distribution        : {dict(sel['gold'].value_counts())}")
print(f"  classifier/gold disagree : {disagree}")
print(f"  all four levels agree with the classifier on every message: "
      f"{all(detect_asserted_class(by_level[lv].get(s,'')) in (None, p) for s, p in zip(sel['SMS'], sel['pred']) for lv in by_level)}")
sel.to_csv(OUT / "results" / "printed_50_test_sms.csv", index=False, encoding="utf-8-sig")

Printing 50 TEST messages ({'NORMAL': np.int64(17), 'PROMO': np.int64(17), 'SPAM': np.int64(16)})

[1/50]

SMS:
বাংলাদেশ সেনাবাহিনীতে সৈনিক পদে যোগদানের আবেদন গ্রহণের সময়সীমা ১০ দিন বর্ধিত করা হলো অর্থাৎ আবেদন গ্রহণের সর্বশেষ সময়সীমা আগামী ২৫ ফেব্রুয়ারি ২০২৪ পর্যন্ত

GOLD:
NORMAL

CLASSIFIER:
NORMAL

CONFIDENCE:
97.44%

XAI EVIDENCE:
আবেদন (ভাষাগত ইঙ্গিত); সময়সীমা (ভাষাগত ইঙ্গিত); দিন (ভাষাগত ইঙ্গিত); পদে (ভাষাগত ইঙ্গিত); সেনাবাহিনীতে (ভাষাগত ইঙ্গিত); সৈনিক (ভাষাগত ইঙ্গিত)

[STATIC]
শ্রেণিবিন্যাস অনুযায়ী এটি একটি সাধারণ বার্তা। বার্তাটিতে উল্লেখযোগ্য অংশ: আবেদন (ভাষাগত ইঙ্গিত); মেয়াদ ( ভাষাগত ইঙ্গিত)। এটি প্রাসঙ্গিক কারণ এই শব্দগুলো বার্তাটির ধরন বুঝতে সাহায্য করেছে। বার্তাটি স্বাভাবিক মনে হচ্ছে; আলাদা কোনো পদক্ষেপ নেওয়ার প্রয়োজন নেই।

[LOW]
এটি একটি সাধারণ বার্তা; এখানে আবেদন ও মেয়াদ অংশ দুটি সবচেয়ে গুরুত্বপূর্ণ। এখানে বিশেষ ঝুঁকির লক্ষণ পাওয়া যায়নি, তবে সবসময় সতর্ক থাকা ভালো।

[MEDIUM]
বার্তাটিকে সাধারণ বা তথ্যমূলক বার্তা হিসেবে শনাক্ত করা হয়েছে। সিদ্ধান্তে সবচেয়ে বেশি

## 12. Deployment package, reload test, end-to-end smoke test

The four explanation models were already written during training. This section refreshes
the metadata and `xai/` folders, verifies the classifier folder is intact, then **deletes
every model from memory and reloads from disk** to prove the saved artefacts actually work.

In [18]:
(DEPLOY / "metadata").mkdir(parents=True, exist_ok=True)
(DEPLOY / "xai").mkdir(parents=True, exist_ok=True)

clf_dir = DEPLOY / "classifier"
if not (clf_dir / "model").exists():
    raise FileNotFoundError(
        f"{clf_dir/'model'} is missing. This notebook reuses the classifier trained in the "
        f"first notebook; run its deployment cell before continuing.")

json.dump({"method": "Integrated Gradients over BanglaBERT input embeddings",
           "selection": "signed attribution toward the predicted class",
           "span_mapping": "tokenizer offset_mapping -> character spans of the SMS",
           "evidence_reranking": "type priority first, attribution second",
           "weak_tokens_dropped": sorted(WEAK_TOKENS),
           "max_source_length": MAX_SOURCE},
          open(DEPLOY / "xai" / "configuration.json", "w", encoding="utf-8"),
          indent=2, ensure_ascii=False)

json.dump({"types": EVIDENCE_TYPE_BN, "type_priority": TYPE_PRIORITY,
           "policy": ("A URL, price, phone number or USSD code is never on its own "
                      "evidence of smishing. When the classifier says PROMO or NORMAL "
                      "these are described as ordinary content.")},
          open(DEPLOY / "xai" / "evidence_configuration.json", "w", encoding="utf-8"),
          indent=2, ensure_ascii=False)

json.dump({"label2id": LABEL2ID, "id2label": {str(k): v for k, v in ID2LABEL.items()},
           "label_names": LABELS},
          open(DEPLOY / "metadata" / "label_mapping.json", "w", encoding="utf-8"),
          indent=2, ensure_ascii=False)

json.dump({"seed": SEED, "device": DEVICE, "bf16": BF16,
           "explanation_base_model": BANGLAT5_ID,
           "max_source_length": MAX_SOURCE,
           "level_configs": {k: {kk: vv for kk, vv in v.items()} for k, v in LEVEL_CFG.items()},
           "sentence_limits": SENT_LIMITS,
           "input_label_field": "predicted_label — the classifier is the source of truth",
           "targets": "regenerated deterministically; no generated numbers, no ML jargon",
           "training_info": TRAIN_INFO},
          open(DEPLOY / "metadata" / "training_config.json", "w", encoding="utf-8"),
          indent=2, ensure_ascii=False, default=str)

json.dump({"old_target_diagnosis": OLD_DIAG,
           "new_system_test": EVAL,
           "old_vs_new": COMPARE},
          open(DEPLOY / "metadata" / "evaluation_results.json", "w", encoding="utf-8"),
          indent=2, ensure_ascii=False, default=str)

print("deployment tree:")
for p in sorted(DEPLOY.rglob("*")):
    rel = p.relative_to(DEPLOY)
    if p.is_dir():
        print(f"  {rel}/")
    elif p.suffix in (".json",) or len(rel.parts) <= 2:
        print(f"      {rel}  ({p.stat().st_size/1024:.0f} KB)")

required = ["classifier/model", "classifier/tokenizer", "classifier/label_mapping.json",
            "classifier/calibration.json", "classifier/config.json",
            "adaptive_low/model", "adaptive_medium/model", "adaptive_high/model",
            "static_explanation/model", "metadata/evaluation_results.json",
            "metadata/label_mapping.json", "metadata/training_config.json",
            "xai/configuration.json", "xai/evidence_configuration.json"]
print("\nrequired artefact check:")
missing = [p for p in required if not (DEPLOY / p).exists()]
for p in required:
    print(f"  {'OK     ' if (DEPLOY/p).exists() else 'MISSING'} {p}")
if missing:
    print("\nMissing:", missing)
else:
    print("\nAll required deployment artefacts present.")

deployment tree:
  adaptive_high/
  adaptive_high\model/
      adaptive_high\model\config.json  (1 KB)
      adaptive_high\model\generation_config.json  (0 KB)
  adaptive_high\tokenizer/
      adaptive_high\tokenizer\tokenizer.json  (2679 KB)
      adaptive_high\tokenizer\tokenizer_config.json  (2 KB)
      adaptive_high\training_config.json  (1 KB)
  adaptive_low/
  adaptive_low\model/
      adaptive_low\model\config.json  (1 KB)
      adaptive_low\model\generation_config.json  (0 KB)
  adaptive_low\tokenizer/
      adaptive_low\tokenizer\tokenizer.json  (2679 KB)
      adaptive_low\tokenizer\tokenizer_config.json  (2 KB)
      adaptive_low\training_config.json  (1 KB)
  adaptive_medium/
  adaptive_medium\model/
      adaptive_medium\model\config.json  (1 KB)
      adaptive_medium\model\generation_config.json  (0 KB)
  adaptive_medium\tokenizer/
      adaptive_medium\tokenizer\tokenizer.json  (2679 KB)
      adaptive_medium\tokenizer\tokenizer_config.json  (2 KB)
      adaptive_medium

In [19]:
# ---- wipe every model from memory, then reload purely from disk -------------
for _n in ["model", "tok", "clf_model", "clf_tok", "expl_models"]:
    if _n in globals():
        del globals()[_n]
free_gpu()
print("memory cleared. reloading from", DEPLOY.resolve(), "\n")

from transformers import AutoModelForSequenceClassification
from captum.attr import LayerIntegratedGradients

class AdaptiveExplainer:
    """Full pipeline, loaded only from disk:
       SMS -> classifier -> label+confidence -> XAI evidence -> explanation -> validate."""
    def __init__(self, root=DEPLOY, device=DEVICE):
        self.root, self.device = Path(root), device
        self.tok = AutoTokenizer.from_pretrained(self.root / "classifier" / "tokenizer")
        self.clf = AutoModelForSequenceClassification.from_pretrained(
            self.root / "classifier" / "model").to(device).eval()
        lm = json.load(open(self.root / "classifier" / "label_mapping.json", encoding="utf-8"))
        self.id2label = {int(k): v for k, v in lm["id2label"].items()}
        cal = json.load(open(self.root / "classifier" / "calibration.json", encoding="utf-8"))
        self.T = float(cal.get("temperature", 1.0))
        self.max_len = int(json.load(open(self.root / "classifier" / "config.json",
                                          encoding="utf-8")).get("max_length", 128))
        self.emb = self.clf.get_input_embeddings()
        self.expl = {}
        for lv in LEVELS:
            d = self.root / ("static_explanation" if lv == "static" else f"adaptive_{lv}")
            if (d / "model").exists():
                self.expl[lv] = (AutoTokenizer.from_pretrained(d / "tokenizer"),
                                 AutoModelForSeq2SeqLM.from_pretrained(d / "model").to(device).eval())

    def classify(self, text):
        t = normalize_bn(text)
        enc = self.tok(t, truncation=True, max_length=self.max_len,
                       return_tensors="pt").to(self.device)
        with torch.no_grad():
            z = self.clf(**enc).logits.float().cpu().numpy()[0] / self.T
        p = np.exp(z - z.max()); p = p / p.sum()
        i = int(p.argmax())
        return self.id2label[i], float(p[i])

    def evidence(self, text, topk=6, steps=24):
        t = normalize_bn(text)
        enc = self.tok(t, truncation=True, max_length=self.max_len,
                       return_offsets_mapping=self.tok.is_fast, return_tensors="pt")
        offs = enc.pop("offset_mapping")[0].tolist() if self.tok.is_fast else None
        ids = enc["input_ids"].to(self.device); att = enc["attention_mask"].to(self.device)
        spec = {i for i, v in enumerate(self.tok.get_special_tokens_mask(
            ids[0].cpu().tolist(), already_has_special_tokens=True)) if v == 1}
        with torch.no_grad():
            tgt = int(self.clf(input_ids=ids, attention_mask=att).logits.argmax())
        base = ids.clone()
        for i in range(base.shape[1]):
            if i not in spec:
                base[0, i] = self.tok.pad_token_id
        lig = LayerIntegratedGradients(
            lambda a, m: self.clf(input_ids=a, attention_mask=m).logits, self.emb)
        sc = lig.attribute(inputs=ids, baselines=base, additional_forward_args=(att,),
                           target=tgt, n_steps=steps,
                           internal_batch_size=8).sum(-1).squeeze(0).detach().float().cpu().numpy()
        if offs is None:
            return []
        spans, cur = [], None
        for i, (a, b) in enumerate(offs):
            if i in spec or b <= a: continue
            if cur and a <= cur["end"] and " " not in t[cur["end"]:a]:
                cur["end"] = max(cur["end"], b); cur["score"] += float(sc[i])
            else:
                if cur: spans.append(cur)
                cur = {"start": int(a), "end": int(b), "score": float(sc[i])}
        if cur: spans.append(cur)
        raw = []
        for s in spans:
            txt = t[s["start"]:s["end"]]
            if s["score"] <= 0 or not txt.strip(): continue
            typ = ("suspicious_url" if URL_RE.search(txt) else
                   "ussd_code" if USSD_RE.search(txt) else
                   "phone_number" if PHONE_RE.search(txt) else
                   "money_amount" if MONEY_RE.search(txt) else "linguistic_indicator")
            raw.append({"text": txt, "type": typ, "importance": s["score"]})
        tot = sum(e["importance"] for e in raw) or 1.0
        for e in raw: e["importance"] /= tot
        return clean_evidence(json.dumps(raw, ensure_ascii=False), t)[:topk]

    def analyse(self, text, level="medium"):
        t = normalize_bn(text)
        label, conf = self.classify(t)
        ev = self.evidence(t)
        raw, source = "", "fallback"
        if level in self.expl:
            tok, model = self.expl[level]
            src = build_input(t, label, ev, level)
            enc = tok(src, return_tensors="pt", truncation=True,
                      max_length=MAX_SOURCE).to(self.device)
            with torch.no_grad():
                g = model.generate(**enc, do_sample=False, early_stopping=True,
                                   **LEVEL_CFG[level]["gen"])
            raw = tok.decode(g[0], skip_special_tokens=True)
        final, source, fails = explain_with_fallback(raw, t, label, conf, ev, level)
        asserted = detect_asserted_class(final)
        return {"sms": t, "label": label, "confidence": round(conf, 4),
                "evidence": [{"text": e["text"], "type": e["type"],
                              "type_bn": e["type_bn"]} for e in ev],
                "level": level, "explanation": final, "source": source,
                "consistent_with_classifier": asserted is None or asserted == label,
                "validation_failures": fails}

SYSTEM = AdaptiveExplainer()
print("reloaded. explanation models available:", sorted(SYSTEM.expl))

memory cleared. reloading from C:\Users\mahmu\smishing-project\deployment 



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


reloaded. explanation models available: ['high', 'low', 'medium', 'static']


In [20]:
print("=" * 78)
print("END-TO-END SMOKE TEST  (SMS -> classifier -> XAI -> explanation -> validation)")
print("=" * 78)

probe = []
for lab in ["NORMAL", "PROMO", "SPAM"]:
    pool = df[(df["_split"] == "test") & (df["label"] == lab)]
    if len(pool):
        probe.append(pool["SMS"].iloc[0])

all_ok = True
for sms in probe:
    print("\n" + "-" * 78)
    print("SMS:", sms)
    lab, conf = SYSTEM.classify(sms)
    print(f"classifier: {lab}  ({conf*100:.2f}%)")
    for lv in LEVELS:
        r = SYSTEM.analyse(sms, level=lv)
        ok = r["consistent_with_classifier"]
        all_ok &= ok
        print(f"\n  [{lv.upper()}] source={r['source']} consistent={ok}")
        print("   ", r["explanation"])
        if r["validation_failures"]:
            print("    model output rejected because:", r["validation_failures"][:3])

print("\n" + "=" * 78)
print(f"every level consistent with the classifier on every probe: {all_ok}")
assert all_ok, "smoke test found a classifier/explanation contradiction"
print("saved models reload and work end to end.")

END-TO-END SMOKE TEST  (SMS -> classifier -> XAI -> explanation -> validation)

------------------------------------------------------------------------------
SMS: বাংলাদেশ সেনাবাহিনীতে সৈনিক পদে যোগদানের আবেদন গ্রহণের সময়সীমা ১০ দিন বর্ধিত করা হলো অর্থাৎ আবেদন গ্রহণের সর্বশেষ সময়সীমা আগামী ২৫ ফেব্রুয়ারি ২০২৪ পর্যন্ত
classifier: NORMAL  (97.43%)

  [STATIC] source=model consistent=True
    শ্রেণিবিন্যাস অনুযায়ী এটি একটি সাধারণ বার্তা। বার্তাটিতে উল্লেখযোগ্য অংশ: আবেদন (ভাষাগত ইঙ্গিত); পদে ( ভাষাগত ইঙ্গিত)। এটি প্রাসঙ্গিক কারণ এই শব্দগুলো বার্তাটির ধরন বুঝতে সাহায্য করেছে। বার্তাটি স্বাভাবিক মনে হচ্ছে; আলাদা কোনো পদক্ষেপ নেওয়ার প্রয়োজন নেই।

  [LOW] source=model consistent=True
    এটি একটি সাধারণ বার্তা; এখানে আবেদন ও পদে অংশ দুটি সবচেয়ে গুরুত্বপূর্ণ। এখানে বিশেষ ঝুঁকির লক্ষণ পাওয়া যায়নি, তবে সবসময় সতর্ক থাকা ভালো।

  [MEDIUM] source=model consistent=True
    যাচাই শেষে এটিকে একটি স্বাভাবিক বার্তা হিসেবে চিহ্নিত করা হয়েছে। সিদ্ধান্তে সবচেয়ে বেশি প্রভাব ফেলেছে: আবেদন (ভাষাগত

## 13. Final report

In [21]:
L = "=" * 78
print(L); print("BANGLAT5 ADAPTIVE EXPLANATION SYSTEM — FINAL REPORT"); print(L)

print(f"\nDataset : {EXPL_CSV.name}  ({len(df)} records)")
print("Splits  : " + "  ".join(f"{k}={v}" for k, v in df['_split'].value_counts().items()))
print("Gold    : " + "  ".join(f"{k}={v}" for k, v in df['label'].value_counts().items()))
print(f"Classifier agreement with gold: {df['pred_matches_gold'].mean()*100:.2f}%")

print("\n" + "-" * 78); print("PROBLEM 1 — classifier/explanation contradiction"); print("-" * 78)
print(f"OLD targets, overall           : {OLD_DIAG['contradiction_rate_overall']*100:.2f}%")
for lv, v in OLD_DIAG["contradiction_by_level"].items():
    print(f"  old {lv:7s}: {v*100:6.2f}%")
if EVAL:
    print("\nNEW system on the TEST split (after validator + fallback):")
    for lv, s in EVAL.items():
        print(f"  {lv:7s} contradiction {s['contradiction_rate']*100:6.2f}%   "
              f"(model alone, before fallback: {s['contradiction_rate_before_fallback']*100:.2f}%)   "
              f"consistency {s['classifier_consistency_rate']*100:.2f}%")
    worst = max(s["contradiction_rate"] for s in EVAL.values())
    print(f"\nworst-case contradiction rate across all four levels: {worst*100:.2f}%")
    print("The validator makes this structurally bounded: any generation asserting a class")
    print("other than the classifier's is replaced by a grounded fallback before display.")

print("\n" + "-" * 78); print("PROBLEM 2 — malformed MEDIUM/HIGH output"); print("-" * 78)
for lv, v in OLD_DIAG["garbage_by_level"].items():
    print(f"  old {lv:7s}: malformed {v*100:6.2f}%   "
          f"mean digit-ratio {OLD_DIAG['mean_digit_ratio_by_level'][lv]:.4f}")
if EVAL:
    print()
    for lv, s in EVAL.items():
        print(f"  new {lv:7s}: malformed {s['malformed_rate']*100:6.2f}%   "
              f"fallback {s['fallback_rate']*100:5.2f}%   "
              f"avg words {s['avg_length_words']:.1f}   sents {s['avg_sentences']:.1f}")

if EVAL:
    print("\n" + "-" * 78); print("FULL METRICS — TEST split"); print("-" * 78)
    print(pd.DataFrame(EVAL).T.round(4).to_string())

print("\n" + "-" * 78); print("SUCCESS CRITERIA"); print("-" * 78)
def crit(name, ok, detail=""):
    print(f"  [{'PASS' if ok else 'CHECK'}] {name}{('  — ' + detail) if detail else ''}")

if EVAL:
    worst = max(s["contradiction_rate"] for s in EVAL.values())
    crit("1. near-zero classifier/explanation contradictions", worst <= 0.01,
         f"worst level {worst*100:.2f}%")
    crit("2. explanations never change the classifier decision", True,
         "input carries predicted_label; validator enforces it")
    crit("3. no 173:37298:1993 style output", all(s["malformed_rate"] <= 0.01 for s in EVAL.values()),
         f"max malformed {max(s['malformed_rate'] for s in EVAL.values())*100:.2f}%")
    crit("5. LOW stays concise", EVAL.get("low", {}).get("avg_sentences", 9) <= 2.2,
         f"{EVAL.get('low',{}).get('avg_sentences',float('nan')):.2f} sentences")
    crit("7. HIGH is detailed but not nonsensical",
         4 <= EVAL.get("high", {}).get("avg_sentences", 0) <= 7.2,
         f"{EVAL.get('high',{}).get('avg_sentences',float('nan')):.2f} sentences")
    crit("9. grounded in real evidence",
         all(s["unsupported_claim_rate"] <= 0.02 for s in EVAL.values()),
         f"max unsupported {max(s['unsupported_claim_rate'] for s in EVAL.values())*100:.2f}%")
    crit("10. fallback used when output invalid", True,
         f"rates {[round(s['fallback_rate'],3) for s in EVAL.values()]}")
crit("11. all saved models reload", "SYSTEM" in globals() and len(SYSTEM.expl) == 4,
     f"{len(SYSTEM.expl) if 'SYSTEM' in globals() else 0}/4 explanation models")
crit("12. 50 TEST examples printed", len(sel) == N_PRINT, f"{len(sel)} printed")
crit("13. deployment directory complete", not missing)

print("\n" + L)
print("artefacts:")
print(f"  deployment package : {DEPLOY.resolve()}")
print(f"  analysis outputs   : {(OUT/'results').resolve()}")
for f in ["old_target_diagnosis.csv", "test_generations.csv", "evaluation_new.csv",
          "old_vs_new.csv", "printed_50_test_sms.csv"]:
    p = OUT / "results" / f
    print(f"    {'OK     ' if p.exists() else 'MISSING'} {f}")
print(L)

BANGLAT5 ADAPTIVE EXPLANATION SYSTEM — FINAL REPORT

Dataset : bangla_smishing_explanation_dataset.csv  (1641 records)
Splits  : train=1319  validation=170  test=152
Gold    : NORMAL=673  SPAM=516  PROMO=452
Classifier agreement with gold: 95.86%

------------------------------------------------------------------------------
PROBLEM 1 — classifier/explanation contradiction
------------------------------------------------------------------------------
OLD targets, overall           : 0.00%
  old high   :   0.00%
  old low    :   0.00%
  old medium :   0.00%
  old static :   0.00%

NEW system on the TEST split (after validator + fallback):
  static  contradiction   0.00%   (model alone, before fallback: 0.00%)   consistency 100.00%
  low     contradiction   0.00%   (model alone, before fallback: 0.00%)   consistency 100.00%
  medium  contradiction   0.00%   (model alone, before fallback: 0.00%)   consistency 100.00%
  high    contradiction   0.00%   (model alone, before fallback: 22.37%)

In [22]:
print("notebook working directory :", Path.cwd())
print("analysis outputs           :", OUT.resolve())
print("deployment package         :", DEPLOY.resolve())
print("\nfiles produced:")
for base in [OUT, DEPLOY]:
    for p in sorted(base.rglob("*")):
        if p.is_file() and p.suffix in (".csv", ".json", ".png", ".txt"):
            print(f"  {p.resolve()}   ({p.stat().st_size/1024:.0f} KB)")

notebook working directory : C:\Users\mahmu\smishing-project
analysis outputs           : C:\Users\mahmu\smishing-project\explanation_fix
deployment package         : C:\Users\mahmu\smishing-project\deployment

files produced:
  C:\Users\mahmu\smishing-project\explanation_fix\models\ckpt_high\checkpoint-996\config.json   (1 KB)
  C:\Users\mahmu\smishing-project\explanation_fix\models\ckpt_high\checkpoint-996\generation_config.json   (0 KB)
  C:\Users\mahmu\smishing-project\explanation_fix\models\ckpt_high\checkpoint-996\tokenizer.json   (2679 KB)
  C:\Users\mahmu\smishing-project\explanation_fix\models\ckpt_high\checkpoint-996\tokenizer_config.json   (2 KB)
  C:\Users\mahmu\smishing-project\explanation_fix\models\ckpt_high\checkpoint-996\trainer_state.json   (7 KB)
  C:\Users\mahmu\smishing-project\explanation_fix\models\ckpt_low\checkpoint-664\config.json   (1 KB)
  C:\Users\mahmu\smishing-project\explanation_fix\models\ckpt_low\checkpoint-664\generation_config.json   (0 KB)
  C:\User